# label_pointwise_opus — the three-arm labeler gate

Step 2 / §8 of `docs/reflective_labeler_design.md`. **First API spend (~a few $).** Decides the whole
artifact: can a whole-doc Opus labeler extract more of the 0.899 ceiling than clf-v4's ~0.61?

Fixed candidate set = **clf top-100** (design §8b). Vary only the scorer:
- **clf-v4** — cached `ce_clf_R.npz`, free.
- **Opus no-CoT** — `thinking=disabled`, bare grade (the §7g *losing* config — the control).
- **Opus CoT** — `thinking=adaptive`, reasons then grades (the calibrated-inference bet).

Grades are 0/1/2 matching the qrel gains (2=Eligible, 1=Excluded, 0=Not relevant). Composition
follows §5.4: labeler reorders the top-W; docs below keep clf order (no zeroed tail). One labeling
run over top-100 is read off at W in {25,50,100} by truncation.

## Setup (Colab — CPU is fine; clf is cached, Opus is API)

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q anthropic nest_asyncio pandas tqdm datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, pickle, time, re, asyncio
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval, full_blob, ndcg_at_k
cfg = ExperimentConfig(data_root=DATA_ROOT)
SPLIT = 'trec22'   # the held-out gate
CLAUDE_MODEL = 'claude-opus-4-8'
PRICE_IN, PRICE_OUT = 5.0, 25.0   # $/1M tokens (Opus 4.8)
LABEL_W = 100                      # label clf top-100; read off 25/50 by truncation
PROBE_N = 10                       # run this many topics first (real cost/quality read); None = all 50
# API key: Colab secret 'ANTHROPIC_API_KEY' (or set the env var yourself)
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    assert os.environ.get('ANTHROPIC_API_KEY'), 'set ANTHROPIC_API_KEY'
print('key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))

## Load corpus, qrels, and the clf top-100 candidate set

In [ ]:
corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
ev = load_eval(cfg, [SPLIT])[SPLIT]
rel, topic2text = ev['rel_dict'], ev['topic2text']
topics = [t for t in rel if t in topic2text]
if PROBE_N:
    topics = topics[:PROBE_N]
    print(f'PROBE: running {len(topics)} topics — set PROBE_N=None and re-run to finish the rest (resumes)')

pool = json.load(open(cfg.path('data/pool_R.json')))[SPLIT]
# clf P(relevant) over the pool from the cached cross-encoder features.
ce = np.load(cfg.ce_cache_path('clf'), allow_pickle=True)['d'].item()   # (s,t,d) -> (P_rel, P_partial)
clf_scores = {t: {d: ce[(SPLIT, t, d)][0] for d in pool[t] if (SPLIT, t, d) in ce} for t in topics}
clf_order  = {t: sorted(pool[t], key=lambda d: clf_scores[t].get(d, -1.0), reverse=True) for t in topics}
print(f'{len(topics)} topics | mean pool {np.mean([len(pool[t]) for t in topics]):.0f} | '
      f'labeling clf top-{LABEL_W} = {len(topics)*LABEL_W} calls/arm')

## The graded labeler — prompt + async driver

Whole-doc `full_blob` (all fields), graded 0/1/2. CoT reasons then emits `GRADE: N`; no-CoT emits only
`GRADE: N` (thinking disabled + final-answer-only, per the Opus-4.8 leak note).

In [ ]:
SYSTEM = ('You are a clinical trial relevance assessor. Given a patient case and a clinical trial, '
          'grade the trial 0-2 for that patient.')

def make_prompt(topic_text, doc_text, arm):
    p = (f'Patient case:\n{topic_text}\n\nClinical trial:\n{doc_text}\n\n'
         'Grade this trial for this patient:\n'
         '2 = Eligible: trial targets the patient\'s condition AND the patient appears to meet its criteria.\n'
         '1 = Excluded: trial targets the condition BUT the patient appears excluded by its criteria.\n'
         '0 = Not relevant: trial does not target the patient\'s condition.\n\n')
    if arm == 'cot':
        p += ('Reason step by step. Where a criterion is not explicitly addressed in the case, make a '
              'calibrated clinical inference rather than abstaining. Then end with exactly: GRADE: N')
    else:
        p += 'Respond with only: GRADE: N   (and nothing else).'
    return p

def parse_grade(text):
    m = re.search(r'GRADE:\s*([012])', text)
    if m: return int(m.group(1))
    m = re.findall(r'[012]', text)      # fallback: last bare 0/1/2
    return int(m[-1]) if m else None

In [ ]:
import anthropic, nest_asyncio
nest_asyncio.apply()
aclient = anthropic.AsyncAnthropic()
MAX_CONCURRENT = 15

async def grade_one(topic_text, doc_text, arm, sem):
    kw = dict(model=CLAUDE_MODEL, system=SYSTEM,
              messages=[{'role': 'user', 'content': make_prompt(topic_text, doc_text, arm)}])
    if arm == 'cot':
        kw.update(thinking={'type': 'adaptive', 'display': 'summarized'},
                  output_config={'effort': 'medium'}, max_tokens=4096)
    else:
        # 64 (not 16) so a short leaked preamble on the thinking-off arm still reaches 'GRADE:'
        kw.update(thinking={'type': 'disabled'}, max_tokens=64)
    async with sem:
        for attempt in range(4):
            try:
                msg = await aclient.messages.create(**kw)
                txt = ''.join(b.text for b in msg.content if b.type == 'text')
                g = parse_grade(txt)
                # ok=False when the grade couldn't be parsed -> defaulted to 0 (control-integrity signal)
                return (g if g is not None else 0, msg.usage.input_tokens, msg.usage.output_tokens, g is not None)
            except Exception as e:
                if attempt == 3:
                    print('give up:', repr(e)[:120]); return 0, 0, 0, False
                await asyncio.sleep(2 ** attempt)

async def grade_topic(t, docs, arm):
    sem = asyncio.Semaphore(MAX_CONCURRENT)
    tasks = [grade_one(topic2text[t], full_blob(id2fields[d], cfg), arm, sem) for d in docs]
    return list(zip(docs, await asyncio.gather(*tasks)))

In [ ]:
# Sanity: one doc, both arms
_t = topics[0]; _d = clf_order[_t][0]
for arm in ['nocot', 'cot']:
    (g, i, o, ok), = [r for _, r in asyncio.run(grade_topic(_t, [_d], arm))]
    print(f'{arm:6s} grade={g} ok={ok}  in={i} out={o}  (gold={rel[_t].get(_d, 0)})')

## Cost estimate before spending (count_tokens on a sample)

The full run issues `len(topics)*LABEL_W*2` Opus calls with whole-doc prompts — **this is tens to a
couple hundred dollars at 50 topics**, not pocket change. Estimate from a real token count first.

In [ ]:
import random
sample = [(t, d) for t in topics for d in clf_order[t][:LABEL_W]]
random.Random(0).shuffle(sample); sample = sample[:20]
in_toks = []
import anthropic as _a
_sync = _a.Anthropic()   # count_tokens via a sync client
for t, d in sample:
    r = _sync.messages.count_tokens(model=CLAUDE_MODEL, system=SYSTEM,
            messages=[{'role': 'user', 'content': make_prompt(topic2text[t], full_blob(id2fields[d], cfg), 'cot')}])
    in_toks.append(r.input_tokens)
mean_in = float(np.mean(in_toks))
n_calls = len(topics) * LABEL_W        # per arm
# assume ~600 output tok/call on CoT (thinking billed as output), ~6 on no-CoT
cot_cost   = n_calls * (mean_in * PRICE_IN + 600 * PRICE_OUT) / 1e6
nocot_cost = n_calls * (mean_in * PRICE_IN + 6   * PRICE_OUT) / 1e6
print(f'mean input tokens/call: {mean_in:.0f} | {n_calls} calls/arm')
print(f'EST for {len(topics)} topics:  no-CoT ~${nocot_cost:.2f}   CoT ~${cot_cost:.2f}   total ~${nocot_cost+cot_cost:.2f}')
print('(CoT output-token guess is rough — the real per-topic $ prints after the run.)')

## Run the two Opus arms over clf top-100 (resumable, cached to Drive)

~5,000 calls/arm. Writes each topic's grades to `data/opus_grades_trec22.jsonl` as it goes; re-running
skips finished (topic, arm) pairs.

In [ ]:
gpath = cfg.path('data/opus_grades_trec22.jsonl')
lpath = cfg.path('data/opus_gate_latency.json')   # (arm,topic) sec — persists across resumes
grades = {'nocot': {}, 'cot': {}}
usage  = {'nocot': [0, 0], 'cot': [0, 0]}
fails  = {'nocot': 0, 'cot': 0}      # parse/API failures defaulted to grade 0 (control-integrity)
lat = json.load(open(lpath)) if os.path.exists(lpath) else {}   # f'{arm}:{t}' -> sec
done = set()
if os.path.exists(gpath):
    for l in open(gpath):
        r = json.loads(l)
        grades[r['arm']][(r['topic_id'], r['doc_id'])] = r['grade']
        usage[r['arm']][0] += r['in_tok']; usage[r['arm']][1] += r['out_tok']
        fails[r['arm']] += 0 if r.get('ok', True) else 1
        done.add((r['arm'], r['topic_id']))
    print('resumed:', {a: sum(1 for x in done if x[0] == a) for a in grades}, 'topics/arm')

with open(gpath, 'a') as f:
    for arm in ['nocot', 'cot']:
        for t in tqdm(topics, desc=f'opus-{arm}'):
            if (arm, t) in done:
                continue
            t0 = time.time()
            res = asyncio.run(grade_topic(t, clf_order[t][:LABEL_W], arm))
            lat[f'{arm}:{t}'] = time.time() - t0
            for d, (g, i, o, ok) in res:
                grades[arm][(t, d)] = g; usage[arm][0] += i; usage[arm][1] += o
                fails[arm] += 0 if ok else 1
                f.write(json.dumps({'arm': arm, 'topic_id': t, 'doc_id': d,
                                    'grade': g, 'in_tok': i, 'out_tok': o, 'ok': ok}) + '\n')
            f.flush(); json.dump(lat, open(lpath, 'w'))
print('done. tokens:', usage, '| defaulted-to-0 (parse/API fails):', fails)

## Results — the accuracy x cost x latency table

NDCG@10 at each labeler width W; docs below W keep clf order (§5.4). clf row = the free baseline the
Opus arms must beat. Cost/latency are per-topic; clf is $0.

In [ ]:
def compose(t, W, garm):
    top = clf_order[t][:W]
    tail = clf_order[t][W:]            # clf order preserved below W (no zeroed tail)
    top_sorted = sorted(top, key=lambda d: (garm.get((t, d), 0), clf_scores[t].get(d, 0)), reverse=True)
    return top_sorted + tail

def ndcg_arm(garm, W):
    return float(np.mean([ndcg_at_k(compose(t, W, garm), rel[t]) for t in topics]))

WS = [25, 50, 100]
rows = []
# clf baseline: no labeler, straight clf order
rows.append({'arm': 'clf-v4 (free)', **{f'ndcg@10 (W={W})': round(float(np.mean([ndcg_at_k(clf_order[t], rel[t]) for t in topics])), 4) for W in WS},
             '$/topic': 0.0, 'sec/topic': None})
for arm in ['nocot', 'cot']:
    ntop = len(topics)
    cost = (usage[arm][0] * PRICE_IN + usage[arm][1] * PRICE_OUT) / 1e6 / ntop
    sec = np.mean([lat[k] for k in lat if k.startswith(arm + ':')]) if any(k.startswith(arm + ':') for k in lat) else None
    n_arm = sum(1 for (t, d) in grades[arm])
    fail_rate = fails[arm] / max(n_arm, 1)
    rows.append({'arm': f'opus-{arm}',
                 **{f'ndcg@10 (W={W})': round(ndcg_arm(grades[arm], W), 4) for W in WS},
                 '$/topic': round(cost, 4), 'sec/topic': round(sec, 1) if sec else None,
                 'fail%': round(100 * fail_rate, 1)})
res_df = pd.DataFrame(rows)
if fails['nocot'] / max(sum(1 for _ in grades['nocot']), 1) > 0.02:
    print('WARNING: no-CoT parse-failure rate > 2% — control may be truncation-compromised; raise max_tokens and rerun that arm')
res_df

In [ ]:
# Reference lines + a peek at grade distribution vs gold (calibration).
print('SOTA (h2oloo) NDCG@10 = 0.6125 | open pipeline = 0.6105 | pool oracle ~0.949 | clf-top100 ceiling 0.899')
for arm in ['nocot', 'cot']:
    import collections
    conf = collections.Counter()
    for (t, d), g in grades[arm].items():
        conf[(rel[t].get(d, 0), g)] += 1
    print(f'\n{arm} gold->pred counts:', dict(sorted(conf.items())))
res_df.to_csv(cfg.path('data/gate_results_trec22.csv'), index=False)
print('\nwrote data/gate_results_trec22.csv')

## Read

- **opus-cot beats clf-v4 and clears ~0.61 at W=100** -> premise holds; build the reflection loop
  (design §5.2-5.4) to push further.
- **opus-cot ties/loses, or only no-CoT loses** -> the CoT-vs-no-CoT gap localizes the cause (§7g);
  a null result folds into the negative-results paper (deliverable #1).
- The **gold->pred counts** show where the labeler errs (2<->1 vs 2<->0/1<->0) — this seeds what the
  reflection stage would target, and flags whether 2<->1 qrel noise dominates (design §5.2).